# 1. Environment Setup & Data Loading

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
url = 'https://raw.githubusercontent.com/hduperera/Datasets/refs/heads/main/CC%20Dataset%20for%20clustering.csv'
df = pd.read_csv(url)
df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [6]:

print("Dataset Shape:", df.shape)

Dataset Shape: (8950, 18)


# 2. Preprocessing

In [7]:


# Drop the customer ID column as it is not a numerical feature for clustering
if 'CUST_ID' in df.columns:
    X = df.drop('CUST_ID', axis=1)
else:
    X = df.copy()

# Handle any missing values by imputing with the median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Scale the features (crucial for distance-based clustering algorithms)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# 3. K Means Clustering

In [8]:


# 3. K-Means Clustering
# Setting k=4 as a common starting point for customer segmentation datasets
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
print(f"K-Means Silhouette Score: {kmeans_silhouette:.4f}")


K-Means Silhouette Score: 0.1977


# 4. Agglomerative Clusteringitalicised text

In [9]:


agg_clustering = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels = agg_clustering.fit_predict(X_scaled)
agg_silhouette = silhouette_score(X_scaled, agg_labels)
print(f"Agglomerative Clustering Silhouette Score: {agg_silhouette:.4f}")



Agglomerative Clustering Silhouette Score: 0.1598


# 5. DBSCAN Clustering

In [10]:
dbscan = DBSCAN(eps=2.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_dbscan = list(dbscan_labels).count(-1)

print(f"DBSCAN estimated number of clusters: {n_clusters_dbscan}")
print(f"DBSCAN estimated number of noise points: {n_noise_dbscan}")

if n_clusters_dbscan > 1:
    dbscan_silhouette = silhouette_score(X_scaled, dbscan_labels)
    print(f"DBSCAN Silhouette Score: {dbscan_silhouette:.4f}")
else:
    print("DBSCAN did not form multiple valid clusters. Try tuning 'eps' or 'min_samples'.")

# Add cluster labels back to the original dataframe for analysis
df['KMeans_Cluster'] = kmeans_labels
df['Agg_Cluster'] = agg_labels
df['DBSCAN_Cluster'] = dbscan_labels

DBSCAN estimated number of clusters: 1
DBSCAN estimated number of noise points: 341
DBSCAN did not form multiple valid clusters. Try tuning 'eps' or 'min_samples'.
